# Inspect clinic.db

In [ ]:
%load_ext autoreload
%autoreload 2

import sqlite3
from pathlib import Path

import pandas as pd

# notebook lives in root/notebooks; the db lives in root/data
DB_PATH = Path("../data/clinic.db")

assert DB_PATH.exists(), f"Could not find {DB_PATH.resolve()} - update DB_PATH above"

conn = sqlite3.connect(DB_PATH)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Tables in the database

In [ ]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)["name"].tolist()
tables

## Schema + row count per table

In [ ]:
for table in tables:
    schema = pd.read_sql(f"PRAGMA table_info({table})", conn)
    row_count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]
    print(f"--- {table} ({row_count} rows) ---")
    display(schema[["name", "type"]])
    print()

## Preview each table

In [ ]:
dfs = {table: pd.read_sql(f"SELECT * FROM {table}", conn) for table in tables}

for table, df in dfs.items():
    print(f"--- {table} ---")
    display(df.head(10))
    print()

## Quick sanity checks

Confirms every canceled appointment has a valid patient, and every waitlist/message row points to a real patient.

In [ ]:
patient_ids = set(dfs["patients"]["patient_id"])

orphan_calendar = dfs["calendar"][~dfs["calendar"]["patient_id"].isin(patient_ids)]
orphan_waitlist = dfs["waitlist"][~dfs["waitlist"]["patient_id"].isin(patient_ids)]
orphan_messages = dfs["messages"][~dfs["messages"]["patient_id"].isin(patient_ids)]

print(f"Orphan calendar rows: {len(orphan_calendar)}")
print(f"Orphan waitlist rows: {len(orphan_waitlist)}")
print(f"Orphan message rows: {len(orphan_messages)}")
print(
    f"Canceled appointments (pending agent work): {(dfs['calendar']['status'] == 'canceled').sum()}"
)

In [ ]:
conn.close()